In [41]:
import torch
from torch import nn
from src.data.dataset import MVTecDataset
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [42]:
ROOT_DIR = 'D:/Unsupervised-Anomaly-Detection-Using-Autoencoder-Architecture/data'
LEARNING_RATE = 1e-3
EPOCHS = 50
BATCH_SIZE = 32
LATENT_DIM = 512        # Care for overfitting, may have to be turned down
BOTTLE_MEAN = [0.5354132056236267, 0.531684160232544, 0.5479597449302673]
BOTTLE_STD = [0.3656117916107178, 0.3659615218639374, 0.34921830892562866]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [43]:
train_test_transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor(),
    transforms.Normalize(mean=BOTTLE_MEAN, std=BOTTLE_STD)
])

mask_transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor()
])

In [44]:
train_data = MVTecDataset(root_dir=ROOT_DIR,
                        category='bottle', 
                        mode='train', 
                        transform=train_test_transform, 
                        mask_transform=None)

test_data = MVTecDataset(root_dir=ROOT_DIR, 
                        category='bottle', 
                        mode='test', 
                        transform=train_test_transform,
                        mask_transform=mask_transform)

train_dataloader = DataLoader(
    dataset=train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True,
    num_workers=2
)

test_dataloader = DataLoader(
    dataset=test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True,
    num_workers=2
)

In [ ]:
class BottleAutoencoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=(3, 3),
                stride=2,
                padding=1
            ),

            nn.LeakyReLU(inplace=True),

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=(3, 3),
                stride=2,
                padding=1,
            ),

            nn.LeakyReLU(inplace=True),

            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=(3, 3),
                stride=2,
                padding=1
            ),

            nn.LeakyReLU(inplace=True),

            nn.Conv2d(
                in_channels=128,
                out_channels=256,
                kernel_size=(3, 3),
                stride=2,
                padding=1
            ),

            nn.LeakyReLU(inplace=True),

            nn.Conv2d(
                in_channels=256,
                out_channels=512,
                kernel_size=(3, 3),
                stride=2,
                padding=1
            ),
            nn.LeakyReLU(inplace=True),

            nn.Conv2d(
                in_channels=512,
                out_channels=64,
                kernel_size=(1, 1),
                stride=1,
                padding=0
            ),
            nn.LeakyReLU(inplace=True),

            nn.Flatten(),
            nn.Linear(64 * 8 * 8, latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64 * 8 * 8),
            nn.Unflatten(dim=1, unflattened_size=(64, 8, 8)),

            nn.Conv2d(
                in_channels=64,
                out_channels=512,
                kernel_size=(1, 1),
                stride=1,
                padding=0
            ),
            nn.LeakyReLU(inplace=True),

            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(
                in_channels=512,
                out_channels=256,
                kernel_size=(3, 3),
                stride=1,
                padding=1
            ),
            nn.LeakyReLU(inplace=True),

            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(
                in_channels=256,
                out_channels=128,
                kernel_size=(3, 3),
                stride=1, 
                padding=1
            ),
            nn.LeakyReLU(inplace=True),

            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(
                in_channels=128,
                out_channels=64,
                kernel_size=(3, 3),
                stride=1,
                padding=1
            ),
            nn.LeakyReLU(inplace=True),

            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(
                in_channels=64,
                out_channels=32,
                kernel_size=(3, 3),
                stride=1,
                padding=1
            ),
            nn.LeakyReLU(inplace=True),

            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(
                in_channels=32,
                out_channels=3,
                kernel_size=(3, 3),
                stride=1,
                padding=1
            ),
            
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [46]:
model = BottleAutoencoder(latent_dim=LATENT_DIM)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = ReduceLROnPlateau(
    optimizer=optimizer,
    mode='min',
    factor=0.05,
    patience=5
)

In [47]:
batch = next(iter(train_dataloader))
if isinstance(batch, torch.Tensor):
    print("DataLoader zwraca TYLKO tensor o kształcie:", batch.shape)
elif isinstance(batch, (list, tuple)):
    print(f"DataLoader zwraca listę/krotkę o długości {len(batch)}")

d:\Unsupervised-Anomaly-Detection-Using-Autoencoder-Architecture\.venv\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


DataLoader zwraca listę/krotkę o długości 4


In [48]:
model.train()
for epoch in range(EPOCHS):
    train_loss = 0.0
    for images, *_ in train_dataloader:
        images = images.to(DEVICE)
        outputs = model(images)

        loss = criterion(outputs, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    scheduler.step(train_loss)

    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {train_loss/len(train_dataloader.dataset):.4f}")
    

Epoch [1/50], Loss: 0.8665
Epoch [2/50], Loss: 0.4732
Epoch [3/50], Loss: 0.4567
Epoch [4/50], Loss: 0.4512
Epoch [5/50], Loss: 0.4549
Epoch [6/50], Loss: 0.4560


KeyboardInterrupt: 